In [ ]:
# 1. Install required packages (100% Offline Mode - including z3-solver)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 768            # Maximum sequence length
BATCH_SIZE = 16            # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 2  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 1                  # Number of training epochs per run
LEARNING_RATE = 2e-4        # Learning rate
OUTPUT_DIR_AUG = "/kaggle/working/results_augmented"          # Output directory for Augmented Run
OUTPUT_DIR_NO_AUG = "/kaggle/working/results_no_augmentation"  # Output directory for No-Augmentation Run

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load NL -> FOL Translation Datasets (Paths left blank per user request)
import os
import json

merged_path = ""  # Fill path to merged_valid.json (augmented)
no_aug_path = ""  # Fill path to merged_valid_no_augmentation.json (no augmentation)

def load_translation_dataset(path):
    if not path or not os.path.exists(path):
        print(f"Warning: Path '{path}' is empty or does not exist. Skipping load.")
        return []
    samples = []
    seen_premises = set()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        samples.append({"premises-NL": nl_list, "premises-FOL": fol_list})
    print(f"Loaded {len(samples)} unique translation samples from {os.path.basename(path)}")
    return samples

raw_samples_aug = load_translation_dataset(merged_path)
raw_samples_no_aug = load_translation_dataset(no_aug_path)


In [ ]:
# 4. Initialize Tokenizer and Load Helper Functions
import os
import sys
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=attn_impl,
        local_files_only=True
)
    model.config.use_cache = False
    return model

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 5. Helper Function to Format Dataset (Chat Template) and split Train/Val
import json
import re
import random
from datasets import Dataset

# Define prompt templates for flat JSON list output with strict count constraint
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n"
    "You must output EXACTLY the same number of formulas as the input premises. Do not skip any premises or merge them.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following {num_premises} premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{premises}\n\n"
    "Return a JSON list of exactly {num_premises} strings containing the formulas, in the exact same order."
)

def normalize_fol(formula):
    # Standardize spaces and keep logic keywords
    keywords = {"ForAll", "Exists", "AND", "OR", "NOT", "In", "->", "<->", "=", "!=", ">=", "<=", ">", "<"}
    # Extract all word-like identifiers
    words = re.findall(r"\b[A-Za-z_][A-Za-z0-9_-]*\b", formula)
    word_map = {}
    counter = 0
    normalized = formula
    for w in words:
        if w in keywords or w.isdigit() or w.startswith("VAR_"):
            continue
        if w not in word_map:
            word_map[w] = f"VAR_{counter}"
            counter += 1
        normalized = re.sub(r"\b" + re.escape(w) + r"\b", word_map[w], normalized)
    return normalized

def get_structure_key(sample):
    fol_list = sample.get("premises-FOL", [])
    normalized_list = [normalize_fol(f) for f in fol_list]
    return "||".join(normalized_list)

def format_hf_dataset_only(raw_samples, tokenizer):
    if not raw_samples:
        return None
    
    formatted_samples = []
    for item in raw_samples:
        nl_list = item["premises-NL"]
        fol_list = item["premises-FOL"]
        
        # Format premises as a numbered list
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        # Render user prompt template with count constraint
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        
        # Assistant response is a flat JSON list of FOL formulas
        assistant_response = json.dumps(fol_list, indent=2)
        
        formatted_samples.append({
            "messages": [
                {"role": "system", "content": system_prompt_template},
                {"role": "user", "content": user_prompt.strip()},
                {"role": "assistant", "content": assistant_response.strip()}
            ]
        })

    dataset = Dataset.from_list(formatted_samples)

    def apply_template(batch):
        texts = []
        for messages in batch["messages"]:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return {"text": texts}

    dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])
    return dataset

def prepare_hf_dataset(raw_samples, tokenizer, test_size=0.1, seed=42):
    if not raw_samples:
        return None, None
        
    # Group samples by their structure key to prevent structural leakage
    groups = {}
    for sample in raw_samples:
        key = get_structure_key(sample)
        if key not in groups:
            groups[key] = []
        groups[key].append(sample)
        
    # Sort keys to be 100% deterministic before shuffling
    unique_keys = sorted(list(groups.keys()))
    
    # Shuffle keys deterministically using a local random generator with a fixed seed
    rng = random.Random(seed)
    rng.shuffle(unique_keys)
    
    # Split keys
    split_idx = int(len(unique_keys) * (1 - test_size))
    train_keys = set(unique_keys[:split_idx])
    val_keys = set(unique_keys[split_idx:])
    
    train_samples = []
    val_samples = []
    for key in unique_keys:
        if key in val_keys:
            val_samples.extend(groups[key])
        else:
            train_samples.extend(groups[key])
            
    # Format to Hugging Face datasets
    train_dataset = format_hf_dataset_only(train_samples, tokenizer)
    val_dataset = format_hf_dataset_only(val_samples, tokenizer)
    
    print(f"Group Split size - Train: {len(train_dataset)}, Val: {len(val_dataset)} (seed={seed})")
    return train_dataset, val_dataset


In [ ]:
# 6. SFT Training Setup and Callbacks
import os
import glob
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

# Custom ReduceLROnPlateau Callback for SFTTrainer
class ReduceLROnPlateauCallback(TrainerCallback):
    def __init__(self, patience=1, factor=0.5, min_lr=1e-6):
        self.patience = patience
        self.factor = factor
        self.min_lr = min_lr
        self.best_loss = float('inf')
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            if eval_loss < self.best_loss:
                self.best_loss = eval_loss
                self.bad_epochs = 0
            else:
                self.bad_epochs += 1
                if self.bad_epochs >= self.patience:
                    self.bad_epochs = 0
                    optimizer = kwargs.get("optimizer")
                    if optimizer is not None:
                        for param_group in optimizer.param_groups:
                            old_lr = param_group['lr']
                            new_lr = max(old_lr * self.factor, self.min_lr)
                            param_group['lr'] = new_lr
                            print(f"\n[ReduceLROnPlateau] Eval loss did not improve for {self.patience} epoch(s). Reducing learning rate from {old_lr:.2e} to {new_lr:.2e}.\n")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

def train_model(train_dataset, val_dataset, output_dir):
    clean_memory()
    
    # Mathematically exact, dynamic warmup steps calculation based on actual dataset size
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * EPOCHS
    
    # Target exactly 3% warmup steps of total training steps (HF v5.2 compliant integer steps)
    warmup_steps = max(1, int(total_steps * 0.03))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    print("Initializing a new PEFT adapter...")
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()
    
    # Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
        warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        neftune_noise_alpha=5.0,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )
    
    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(response_template=response_template, tokenizer=tokenizer)
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback(), ReduceLROnPlateauCallback(patience=1, factor=0.5)]
    )
    
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    # Clean up model & trainer from memory
    del trainer
    del model
    del base_model
    clean_memory()


In [ ]:
# 6.1 Run Training: Augmented Dataset (Run 1)
if not raw_samples_aug:
    print("Skipping Run 1 training because raw_samples_aug is empty. Please set merged_path.")
else:
    print("=== Starting Training Run 1: Augmented Dataset ===")
    train_dataset_aug, val_dataset_aug = prepare_hf_dataset(raw_samples_aug, tokenizer)
    train_model(train_dataset_aug, val_dataset_aug, OUTPUT_DIR_AUG)


In [ ]:
# 6.2 Run Training: No-Augmentation Dataset (Run 2)
if not raw_samples_no_aug:
    print("Skipping Run 2 training because raw_samples_no_aug is empty. Please set no_aug_path.")
else:
    print("=== Starting Training Run 2: No-Augmentation Dataset ===")
    train_dataset_no_aug, val_dataset_no_aug = prepare_hf_dataset(raw_samples_no_aug, tokenizer)
    train_model(train_dataset_no_aug, val_dataset_no_aug, OUTPUT_DIR_NO_AUG)


In [ ]:
# 7. Self-Contained First-Order Logic Parser & Metric Utilities
# Embedded directly from src/logic/reasoning/parser.py for 100% Offline Kaggle Compatibility

"""Parse simple FOL strings into Z3 expressions.

Assumptions:
- Quantifiers use `ForAll(x, ...)` / `Exists(x, ...)`.
- Boolean operators: `AND`, `OR`, `NOT`, `->`, `<->`.
- Predicates are functions returning Bool, e.g., `P(x)` or `R(a,b)`.
- Comparisons supported: `=`, `!=`, `>=`, `<=`, `>`, `<`.
"""


import re
from typing import Dict, Iterable, List, Optional, Tuple

import z3
from z3 import (
    And,
    BoolSort,
    Const,
    DeclareSort,
    Exists,
    ForAll,
    Function,
    IntSort,
    IntVal,
    Not,
    Or,
    RealSort,
    RealVal,
    StringVal,
    BoolRef,
    ExprRef,
)

_TOKEN_RE = re.compile(
    r"\s*(->|<->|AND|OR|NOT|IN|ForAll|Exists|>=|<=|!=|=|>|<|\(|\)|,|\+|-|\d+\.\d+|\d+|'[^']*'|[^\W\d][\w-]*)"
)


class Z3Symbols:
    def __init__(self, sort: ExprRef) -> None:
        self.sort = sort
        self.consts: Dict[str, ExprRef] = {}
        self.preds: Dict[Tuple[str, int], ExprRef] = {}
        self.funcs: Dict[Tuple[str, int], ExprRef] = {}
        self.numeric_symbols: set[str] = set()

    def get_const(self, name: str, sort: Optional[ExprRef] = None) -> ExprRef:
        if name in self.consts:
            return self.consts[name]
        use_sort = sort
        if use_sort is None:
            use_sort = RealSort() if name in self.numeric_symbols else self.sort
        const = Const(name, use_sort)
        self.consts[name] = const
        return const

    def get_pred(self, name: str, arity: int) -> ExprRef:
        key = (name, arity)
        if key in self.preds:
            return self.preds[key]
        # Upgrades predicates to RealSort functions if they are involved in comparisons
        if name in self.numeric_symbols:
            return self.get_func(name, arity, RealSort())
        pred = Function(name, *([self.sort] * arity), BoolSort())
        self.preds[key] = pred
        return pred

    def get_func(self, name: str, arity: int, sort: Optional[ExprRef] = None) -> ExprRef:
        key = (name, arity)
        if key in self.funcs:
            return self.funcs[key]
        use_sort = sort
        if use_sort is None:
            use_sort = RealSort() if name in self.numeric_symbols else self.sort
        func = Function(name, *([self.sort] * arity), use_sort)
        self.funcs[key] = func
        return func


class TokenStream:
    def __init__(self, text: str) -> None:
        self.tokens = [t for t in _TOKEN_RE.findall(text) if t.strip()]
        self.index = 0

    def peek(self) -> Optional[str]:
        if self.index >= len(self.tokens):
            return None
        return self.tokens[self.index]

    def peek_offset(self, offset: int) -> Optional[str]:
        idx = self.index + offset
        if idx >= len(self.tokens):
            return None
        return self.tokens[idx]

    def next(self) -> Optional[str]:
        tok = self.peek()
        if tok is not None:
            self.index += 1
        return tok

    def expect(self, value: str) -> None:
        tok = self.next()
        if tok != value:
            raise ValueError(f"Expected '{value}', got '{tok}'")


class FolParser:
    def __init__(self, symbols: Z3Symbols) -> None:
        self.symbols = symbols
        self.var_stack: List[Dict[str, ExprRef]] = []

    def parse(self, text: str) -> BoolRef:
        stream = TokenStream(text)
        expr = self._parse_implication(stream)
        return expr

    def _parse_implication(self, stream: TokenStream) -> BoolRef:
        left = self._parse_or(stream)
        tok = stream.peek()
        if tok == "->":
            stream.next()
            right = self._parse_implication(stream)
            return Or(Not(left), right)
        if tok == "<->":
            stream.next()
            right = self._parse_implication(stream)
            return And(Or(Not(left), right), Or(Not(right), left))
        return left

    def _parse_or(self, stream: TokenStream) -> BoolRef:
        left = self._parse_and(stream)
        while stream.peek() == "OR":
            stream.next()
            right = self._parse_and(stream)
            left = Or(left, right)
        return left

    def _parse_and(self, stream: TokenStream) -> BoolRef:
        left = self._parse_not(stream)
        while stream.peek() == "AND":
            stream.next()
            right = self._parse_not(stream)
            left = And(left, right)
        return left

    def _parse_not(self, stream: TokenStream) -> BoolRef:
        if stream.peek() == "NOT":
            stream.next()
            return Not(self._parse_not(stream))
        return self._parse_atom(stream)

    def _parse_atom(self, stream: TokenStream) -> BoolRef:
        tok = stream.peek()
        if tok is None:
            raise ValueError("Unexpected end of input")
        if tok == "(":
            stream.next()
            expr = self._parse_implication(stream)
            stream.expect(")")
            return expr
        if tok in ("ForAll", "Exists"):
            return self._parse_quantifier(stream)
        start_index = stream.index
        term = self._parse_term(stream)
        comp = stream.peek()
        if comp == "IN":
            stream.next()
            right = self._parse_term(stream)
            pred = self.symbols.get_pred("In", 2)
            return pred(term, right)
        if comp in ("=", "!=", ">=", "<=", ">", "<"):
            numeric = comp in (">=", "<=", ">", "<")
            if comp in ("=", "!="):
                next_tok = stream.peek_offset(1)
                if next_tok is not None and re.fullmatch(r"\d+(?:\.\d+)?", next_tok):
                    numeric = True
            if numeric and isinstance(term, BoolRef):
                stream.index = start_index
                term = self._parse_term(stream, prefer_numeric=True)
                comp = stream.peek()
            stream.next()
            right = self._parse_term(stream, prefer_numeric=numeric)
            return self._build_comparison(comp, term, right)
        if not isinstance(term, BoolRef):
            raise ValueError("Predicate expected, got term")
        return term

    def _parse_quantifier(self, stream: TokenStream) -> BoolRef:
        quant = stream.next()
        stream.expect("(")
        var_name = stream.next()
        if var_name is None:
            raise ValueError("Missing quantified variable")
        stream.expect(",")
        var = Const(var_name, self.symbols.sort)
        self.var_stack.append({var_name: var})
        body = self._parse_implication(stream)
        self.var_stack.pop()
        stream.expect(")")
        if quant == "ForAll":
            return ForAll([var], body)
        return Exists([var], body)

    def _parse_term(self, stream: TokenStream, prefer_numeric: bool = False) -> ExprRef:
        left = self._parse_simple_term(stream, prefer_numeric=prefer_numeric)
        while stream.peek() in ("+", "-"):
            op = stream.next()
            right = self._parse_simple_term(stream, prefer_numeric=True)
            if op == "+":
                left = left + right
            else:
                left = left - right
        return left

    def _parse_simple_term(self, stream: TokenStream, prefer_numeric: bool = False) -> ExprRef:
        tok = stream.next()
        if tok is None:
            raise ValueError("Unexpected end of input")
        if tok == "(":
            expr = self._parse_term(stream, prefer_numeric=prefer_numeric)
            stream.expect(")")
            return expr
        if tok.startswith("'") and tok.endswith("'"):
            return StringVal(tok[1:-1])

        # Parse temporal time constants (e.g., Time830AM, Time500PM) as minutes since midnight
        time_match = re.match(r"^Time(\d{1,2})(\d{2})?(AM|PM)$", tok, re.IGNORECASE)
        if time_match:
            hour = int(time_match.group(1))
            minute = int(time_match.group(2)) if time_match.group(2) else 0
            ampm = time_match.group(3).upper()
            if ampm == "PM" and hour < 12:
                hour += 12
            elif ampm == "AM" and hour == 12:
                hour = 0
            minutes = hour * 60 + minute
            return IntVal(minutes)

        # Parse temporal durations (e.g., Duration4Hours, Duration30Minutes) as minutes
        duration_match = re.match(r"^Duration(\d+(?:\.\d+)?)(Hours|Minutes|Days)$", tok, re.IGNORECASE)
        if duration_match:
            value = float(duration_match.group(1))
            unit = duration_match.group(2).lower()
            if unit == "hours":
                minutes = int(value * 60)
            elif unit == "days":
                minutes = int(value * 24 * 60)
            else:
                minutes = int(value)
            return IntVal(minutes)

        if tok.replace(".", "", 1).isdigit():
            if "." in tok:
                return RealVal(tok)
            return IntVal(tok)
        var = self._lookup_var(tok)
        if var is not None:
            return var
        if stream.peek() == "(":
            stream.next()
            args = []
            if stream.peek() != ")":
                while True:
                    args.append(self._parse_term(stream, prefer_numeric=prefer_numeric))
                    if stream.peek() == ",":
                        stream.next()
                        continue
                    break
            stream.expect(")")
            if prefer_numeric:
                func = self.symbols.get_func(tok, len(args), RealSort())
                return func(*args)
            pred = self.symbols.get_pred(tok, len(args))
            return pred(*args)
        if prefer_numeric:
            return self.symbols.get_const(tok, RealSort())
        return self.symbols.get_const(tok)

    def _build_comparison(self, op: str, left: ExprRef, right: ExprRef) -> BoolRef:
        if op == "=":
            return left == right
        if op == "!=":
            return left != right
        if op == ">=":
            return left >= right
        if op == "<=":
            return left <= right
        if op == ">":
            return left > right
        if op == "<":
            return left < right
        raise ValueError(f"Unsupported comparator: {op}")

    def _lookup_var(self, name: str) -> Optional[ExprRef]:
        for scope in reversed(self.var_stack):
            if name in scope:
                return scope[name]
        return None


def parse_formulas(
    formulas: Iterable[str],
    sort_name: str = "U",
) -> Tuple[Z3Symbols, List[BoolRef]]:
    """Parse a flat list of first-order logic formulas into Z3 Bool expressions.

    Returns a tuple of (symbols, formula_exprs).
    """
    # Robust pre-scan to identify numeric symbols and prevent sort mismatches
    numeric_symbols = set()
    for f in formulas:
        # Temporarily replace logical arrows to prevent false matching of '-' or '>'
        f_temp = f.replace("<->", " BICOND ").replace("->", " IMPLIES ")
        
        # 1. Match identifiers on the left of inequality or arithmetic operators
        left_matches = re.finditer(r"\b([A-Za-z_][A-Za-z0-9_-]*)\s*(?:\([^()]*\))?\s*(?:>=|<=|>|<|\+|-)\b", f_temp)
        for m in left_matches:
            name = m.group(1)
            if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                numeric_symbols.add(name)
                
        # 2. Match identifiers on the right of inequality or arithmetic operators
        right_matches = re.finditer(r"\b(?:>=|<=|>|<|\+|-)\s*([A-Za-z_][A-Za-z0-9_-]*)\b", f_temp)
        for m in right_matches:
            name = m.group(1)
            if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                numeric_symbols.add(name)
                
        # 3. Match equality/inequality with numeric or temporal/duration literals on either side
        eq_matches = re.finditer(r"\b([A-Za-z_][A-Za-z0-9_-]*)\s*(?:\([^()]*\))?\s*(?:=|!=)\s*(?:\d+(?:\.\d+)?|Time\d+[A-Za-z]+|Duration\d+[A-Z]+)\b", f_temp)
        for m in eq_matches:
            name = m.group(1)
            if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                numeric_symbols.add(name)
                
        eq_rev_matches = re.finditer(r"\b(?:\d+(?:\.\d+)?|Time\d+[A-Za-z]+|Duration\d+[A-Z]+)\s*(?:=|!=)\s*([A-Za-z_][A-Za-z0-9_-]*)\b", f_temp)
        for m in eq_rev_matches:
            name = m.group(1)
            if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                numeric_symbols.add(name)

    symbols = Z3Symbols(sort=DeclareSort(sort_name))
    symbols.numeric_symbols = numeric_symbols
    
    parser = FolParser(symbols)
    formula_exprs: List[BoolRef] = [parser.parse(f) for f in formulas]

    return symbols, formula_exprs


import threading

# Global lock to ensure thread-safe access to Z3 (which is not thread-safe by default)
_z3_lock = threading.Lock()


def try_parse_fol(formula: str) -> tuple[bool, str]:
    """Try to parse a single FOL formula string into a Z3 expression.

    Returns:
        (True, "")           if the formula parses successfully.
        (False, error_msg)   if the parser raises an exception.

    Used by the translation repair loop to validate each generated formula
    before committing it, without running a full solver check.
    """
    with _z3_lock:
        try:
            parse_formulas([formula])
            return True, ""
        except Exception as exc:
            return False, str(exc)

# =========================================================================
# Metric Utilities using the embedded parser
# =========================================================================
from collections import Counter

def compute_exact_match(pred: str, gt: str) -> bool:
    return pred.strip() == gt.strip()

def z3_ast_equal(e1, e2, var_map=None):
    if var_map is None:
        var_map = {}
    if z3.is_quantifier(e1) and z3.is_quantifier(e2):
        if e1.is_forall() != e2.is_forall():
            return False
        if e1.num_vars() != e2.num_vars():
            return False
        new_map = var_map.copy()
        for i in range(e1.num_vars()):
            v1 = e1.var_name(i)
            v2 = e2.var_name(i)
            new_map[v1] = v2
        return z3_ast_equal(e1.body(), e2.body(), new_map)
    if z3.is_app(e1) and z3.is_app(e2):
        decl1 = e1.decl()
        decl2 = e2.decl()
        name1 = decl1.name()
        name2 = decl2.name()
        if e1.num_args() == 0 and e2.num_args() == 0:
            if name1 in var_map:
                return var_map[name1] == name2
            return name1 == name2
        if name1 != name2:
            return False
        if e1.num_args() != e2.num_args():
            return False
        return all(z3_ast_equal(c1, c2, var_map) for c1, c2 in zip(e1.children(), e2.children()))
    return str(e1) == str(e2)

def compute_ast_match(pred: str, gt: str) -> bool:
    if pred.strip() == gt.strip():
        return True
    try:
        _, exprs = parse_formulas([pred, gt])
        return z3_ast_equal(exprs[0], exprs[1])
    except Exception:
        return False

def compute_logical_equivalence(pred: str, gt: str) -> bool:
    if pred.strip() == gt.strip():
        return True
    try:
        _, exprs = parse_formulas([pred, gt])
        e1, e2 = exprs[0], exprs[1]
        solver = z3.Solver()
        solver.set("timeout", 5000)
        solver.add(z3.Not(e1 == e2))
        return solver.check() == z3.unsat
    except Exception:
        return False

def get_symbols(formula_str):
    try:
        stream = TokenStream(formula_str)
        return [t for t in stream.tokens if t not in ("(", ")", ",", " ")]
    except Exception:
        return [t for t in re.findall(r"\w+", formula_str)]

def compute_symbol_accuracy(pred: str, gt: str) -> float:
    pred_tokens = get_symbols(pred)
    gt_tokens = get_symbols(gt)
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    c_pred = Counter(pred_tokens)
    c_gt = Counter(gt_tokens)
    intersection = sum((c_pred & c_gt).values())
    union = sum((c_pred | c_gt).values())
    return intersection / union if union > 0 else 0.0

print("Self-contained logic parser & evaluation metrics loaded successfully!")


In [ ]:
# 7.1 Direct Dataset Ground-Truth Comparison
def compare_datasets_directly(samples_aug, samples_no_aug):
    if not samples_aug or not samples_no_aug:
        print("One or both datasets are empty. Skipping direct comparison.")
        return
        
    map_aug = {"\n".join(s["premises-NL"]): s["premises-FOL"] for s in samples_aug}
    map_no_aug = {"\n".join(s["premises-NL"]): s["premises-FOL"] for s in samples_no_aug}
    
    common_nl = set(map_aug.keys()).intersection(set(map_no_aug.keys()))
    if not common_nl:
        print("No overlapping natural language premises found between the two datasets.")
        return
        
    print(f"Comparing target FOL formulas across {len(common_nl)} overlapping premises...")
    
    exact_matches = 0
    ast_matches = 0
    logical_eqs = 0
    symbol_accs = []
    
    for nl in common_nl:
        fol_aug = map_aug[nl]
        fol_no_aug = map_no_aug[nl]
        
        em_list, ast_list, le_list, sa_list = [], [], [], []
        max_len = max(len(fol_aug), len(fol_no_aug))
        for i in range(max_len):
            f_aug = fol_aug[i] if i < len(fol_aug) else ""
            f_no_aug = fol_no_aug[i] if i < len(fol_no_aug) else ""
            
            em_list.append(compute_exact_match(f_aug, f_no_aug))
            ast_list.append(compute_ast_match(f_aug, f_no_aug))
            le_list.append(compute_logical_equivalence(f_aug, f_no_aug))
            sa_list.append(compute_symbol_accuracy(f_aug, f_no_aug))
            
        exact_matches += sum(em_list) / len(em_list) if em_list else 0
        ast_matches += sum(ast_list) / len(ast_list) if ast_list else 0
        logical_eqs += sum(le_list) / len(le_list) if le_list else 0
        symbol_accs.append(sum(sa_list) / len(sa_list) if sa_list else 0)
        
    num_evaluated = len(common_nl)
    print("\n" + "=" * 50)
    print("   DIRECT GROUND-TRUTH DATASET COMPARISON")
    print("=" * 50)
    print(f"Common Premises Evaluated: {num_evaluated}")
    print(f"Exact Match:              {(exact_matches / num_evaluated) * 100:.2f}%")
    print(f"AST Match (Structural):    {(ast_matches / num_evaluated) * 100:.2f}%")
    print(f"Logical Equivalence:       {(logical_eqs / num_evaluated) * 100:.2f}%")
    print(f"Symbol Accuracy (Overlap):  {(sum(symbol_accs) / num_evaluated) * 100:.2f}%")
    print("=" * 50)

compare_datasets_directly(raw_samples_aug, raw_samples_no_aug)


In [ ]:
# 8. Running Inference Test and Side-by-Side Comparison
import torch
from peft import PeftModel
import json

test_nl_premises = [
    "All humans are mortal.",
    "Socrates is a human."
]

test_nl_content = ""
for i, nl in enumerate(test_nl_premises, start=1):
    test_nl_content += f"{i}. {nl}\n"

test_user_prompt = user_prompt_template.format(num_premises=len(test_nl_premises), premises=test_nl_content.strip())

messages = [
    {"role": "system", "content": system_prompt_template},
    {"role": "user", "content": test_user_prompt.strip()}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

def run_inference_on_model(adapter_dir):
    if not os.path.exists(adapter_dir) or not os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
        return "Model adapter not found/trained yet."
    
    clean_memory()
    base_model = load_base_model()
    print(f"Loading adapter weights from: {adapter_dir}")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)
    
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    del model
    del base_model
    clean_memory()
    return response_text

print("\n--- NL Premises ---")
print(test_nl_content.strip())
print("--------------------------------------------------")

print("\n--- Running Inference for Run 1: Augmented Model ---")
output_aug = run_inference_on_model(OUTPUT_DIR_AUG)
print(output_aug)
print("--------------------------------------------------")

print("\n--- Running Inference for Run 2: No-Augmentation Model ---")
output_no_aug = run_inference_on_model(OUTPUT_DIR_NO_AUG)
print(output_no_aug)
print("--------------------------------------------------")


In [ ]:
# 9. Comprehensive Side-by-Side Model Evaluation
import json
import os
import torch
from peft import PeftModel

def evaluate_model_on_val(adapter_dir, val_dataset, num_samples=30):
    if not val_dataset:
        return {"exact_match": 0.0, "ast_match": 0.0, "logical_equivalence": 0.0, "symbol_accuracy": 0.0}
        
    if not os.path.exists(adapter_dir) or not os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
        print(f"Model adapter not found at {adapter_dir}. Skipping.")
        return None
        
    clean_memory()
    base_model = load_base_model()
    print(f"Loading adapter weights from: {adapter_dir}")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.to(device)
    model.eval()
    
    subset = val_dataset.select(range(min(num_samples, len(val_dataset))))
    
    exact_matches = 0
    ast_matches = 0
    logical_eqs = 0
    symbol_accs = []
    
    for idx, item in enumerate(subset):
        text = item["text"]
        parts = text.split("<|im_start|>assistant\n")
        if len(parts) < 2:
            continue
        prompt = parts[0] + "<|im_start|>assistant\n"
        gt_str = parts[1].replace("<|im_end|>", "").replace(tokenizer.eos_token, "").strip()
        
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)
        
        pred_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        try:
            gt_formulas = json.loads(gt_str)
        except Exception:
            gt_formulas = [gt_str]
            
        try:
            pred_formulas = json.loads(pred_str)
        except Exception:
            pred_formulas = [pred_str]
            
        em_list, ast_list, le_list, sa_list = [], [], [], []
        max_len = max(len(gt_formulas), len(pred_formulas))
        for i in range(max_len):
            gt_f = gt_formulas[i] if i < len(gt_formulas) else ""
            pred_f = pred_formulas[i] if i < len(pred_formulas) else ""
            
            em_list.append(compute_exact_match(pred_f, gt_f))
            ast_list.append(compute_ast_match(pred_f, gt_f))
            le_list.append(compute_logical_equivalence(pred_f, gt_f))
            sa_list.append(compute_symbol_accuracy(pred_f, gt_f))
            
        exact_matches += sum(em_list) / len(em_list) if em_list else 0
        ast_matches += sum(ast_list) / len(ast_list) if ast_list else 0
        logical_eqs += sum(le_list) / len(le_list) if le_list else 0
        symbol_accs.append(sum(sa_list) / len(sa_list) if sa_list else 0)
        
    num_evaluated = len(subset)
    metrics = {
        "exact_match": (exact_matches / num_evaluated) * 100 if num_evaluated else 0.0,
        "ast_match": (ast_matches / num_evaluated) * 100 if num_evaluated else 0.0,
        "logical_equivalence": (logical_eqs / num_evaluated) * 100 if num_evaluated else 0.0,
        "symbol_accuracy": (sum(symbol_accs) / num_evaluated) * 100 if num_evaluated else 0.0,
    }
    
    del model
    del base_model
    clean_memory()
    return metrics

print("\n--- Evaluating Model 1: Augmented Dataset ---")
metrics_aug = None
if "val_dataset_no_aug" in globals() and val_dataset_no_aug:
    metrics_aug = evaluate_model_on_val(OUTPUT_DIR_AUG, val_dataset_no_aug, num_samples=30)
elif "val_dataset_aug" in globals() and val_dataset_aug:
    metrics_aug = evaluate_model_on_val(OUTPUT_DIR_AUG, val_dataset_aug, num_samples=30)

print("\n--- Evaluating Model 2: No-Augmentation Dataset ---")
metrics_no_aug = None
if "val_dataset_no_aug" in globals() and val_dataset_no_aug:
    metrics_no_aug = evaluate_model_on_val(OUTPUT_DIR_NO_AUG, val_dataset_no_aug, num_samples=30)

if metrics_aug or metrics_no_aug:
    print("\n" + "=" * 65)
    print("               SIDE-BY-SIDE MODEL EVALUATION SUMMARY")
    print("=" * 65)
    print(f"{'Metric':<25} | {'Augmented Model':<16} | {'No-Aug Model':<16}")
    print("-" * 65)
    for key in ["exact_match", "ast_match", "logical_equivalence", "symbol_accuracy"]:
        val_aug = f"{metrics_aug[key]:.2f}%" if metrics_aug else "N/A"
        val_no_aug = f"{metrics_no_aug[key]:.2f}%" if metrics_no_aug else "N/A"
        label = key.replace("_", " ").title()
        print(f"{label:<25} | {val_aug:<16} | {val_no_aug:<16}")
    print("=" * 65)
